<a href="https://colab.research.google.com/github/erickcorrea3092/Antobiotics_Kurt_S_aureus/blob/main/1.2_Descriptores_Dionisio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://www.youtube.com/watch?v=9i9SY6Nd1Zw

In [1]:
import sys
import numpy as np

print("Python:", sys.version)
print("NumPy:", np.__version__)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
NumPy: 2.1.3


In [2]:
import scipy
import sklearn

print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)

print("NUMPY + SCIPY + SKLEARN OK")

SciPy: 1.16.3
Scikit-learn: 1.6.1
NUMPY + SCIPY + SKLEARN OK


In [3]:
try:
    import umap
    print("UMAP:", umap.__version__)
    print("UMAP OK")
except Exception as e:
    print("ERROR UMAP:")
    print(type(e).__name__, e)

UMAP: 0.5.12
UMAP OK


In [4]:
# from IPython.utils import io
# import tqdm.notebook
# import os, os.path, sys, random, subprocess
# total = 100
# with tqdm.notebook.tqdm(total=total) as pbar:
#     with io.capture_output() as captured:
#       pbar.update(0)
#       !pip install --upgrade --force-reinstall numpy
#       pbar.update(33)
#       !pip install --upgrade --force-reinstall pandas
#       pbar.update(66)
#       !pip install --upgrade --force-reinstall mordred
#       pbar.update(100)

In [8]:
! pip install rdkit
! pip install mordredcommunity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 6.0 MB/s eta 0:00:00


In [9]:
from rdkit.Chem import AllChem
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
import pandas as pd
import numpy as np
from mordred import Calculator, descriptors

In [10]:
!gdown --id 1YzL6UcBvD0evWZYOe2pRDXXSDGWRGgmJ

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1YzL6UcBvD0evWZYOe2pRDXXSDGWRGgmJ
To: /content/data2.xlsx
100% 64.5k/64.5k [00:00<00:00, 3.26MB/s]


In [11]:
dataset = pd.read_excel('data2.xlsx')
dataset.shape

(1695, 3)

In [12]:
dataset.head()

,Cmpd,SMILES,Activity
0,CHEMBL377507,C/C(=N/NC(=O)c1ccncc1)c1ccc(N/C(S)=N/c2ccccc2)cc1,5.397940
1,CHEMBL64,NNC(=O)c1ccncc1,5.835647
2,CHEMBL204856,CCOc1ccc(N/C(S)=N\c2ccc(/C(C)=N\NC(=O)c3ccncc3...,6.050610
3,CHEMBL204135,COc1ccc(N/C(S)=N\c2ccc(/C(C)=N\NC(=O)c3ccncc3)...,6.327902
4,CHEMBL204068,C/C(=N/NC(=O)c1ccncc1)c1ccc(N/C(S)=N/c2ccc(F)c...,6.309804


In [13]:
def canonical_smiles (smiles):
  mols = [Chem.MolFromSmiles(smi) for smi in smiles]
  smiles = [Chem.MolToSmiles(mol) for mol in mols]
  return smiles

In [14]:
Canon_SMILES = canonical_smiles(dataset.SMILES)
len(Canon_SMILES)

1695

In [15]:
#put the smiles in dataframe
dataset['SMILES'] = Canon_SMILES

In [16]:
#create a list for duplicate SMILES

duplicates_smiles = dataset[dataset['SMILES'].duplicated()]['SMILES'].values
len(duplicates_smiles)

353

In [17]:
#Create a list for duplicate smiles
dataset[dataset['SMILES'].isin(duplicates_smiles)].sort_values(by=['SMILES'])

,Cmpd,SMILES,Activity
623,CHEMBL126,CC(=O)NC[C@H]1CN(c2ccc(N3CCOCC3)c(F)c2)C(=O)O1,5.823909
633,CHEMBL126,CC(=O)NC[C@H]1CN(c2ccc(N3CCOCC3)c(F)c2)C(=O)O1,5.823909
978,CHEMBL126,CC(=O)NC[C@H]1CN(c2ccc(N3CCOCC3)c(F)c2)C(=O)O1,2.829111
1672,CHEMBL5272309,CC(=O)Nc1ccc(N2C=C(C(c3c[nH]c4ccc(Br)cc34)c3c[...,2.782703
1671,CHEMBL5272309,CC(=O)Nc1ccc(N2C=C(C(c3c[nH]c4ccc(Br)cc34)c3c[...,2.305581
...,...,...,...
340,CHEMBL1802198,Oc1c(Br)cc(Br)c2nc3ccccc3nc12,4.602060
638,CHEMBL1802198,Oc1c(Br)cc(Br)c2nc3ccccc3nc12,4.602060
422,CHEMBL1802198,Oc1c(Br)cc(Br)c2nc3ccccc3nc12,4.602060
399,CHEMBL4207593,S=C(NCc1ccccc1)Nc1cccc(-c2nc3ccccc3s2)c1,4.176330


###2.Drop duplicates value

In [18]:
dataset_new = dataset.drop_duplicates(subset=['SMILES'])
len(dataset_new)

1342

In [19]:
dataset_new.reset_index(drop=True, inplace=True)
dataset_new

,Cmpd,SMILES,Activity
0,CHEMBL377507,C/C(=N/NC(=O)c1ccncc1)c1ccc(N/C(S)=N/c2ccccc2)cc1,5.397940
1,CHEMBL64,NNC(=O)c1ccncc1,5.835647
2,CHEMBL204856,CCOc1ccc(N/C(S)=N\c2ccc(/C(C)=N\NC(=O)c3ccncc3...,6.050610
3,CHEMBL204135,COc1ccc(N/C(S)=N\c2ccc(/C(C)=N\NC(=O)c3ccncc3)...,6.327902
4,CHEMBL204068,C/C(=N/NC(=O)c1ccncc1)c1ccc(N/C(S)=N/c2ccc(F)c...,6.309804
...,...,...,...
1337,CHEMBL5289448,O=C(NN1C(=O)CSC1c1ccc([N+](=O)[O-])cc1)c1ccc(N...,2.261465
1338,CHEMBL5271061,O=C1Nc2cc3c(-c4ccccc4)c(-c4ccccc4)oc(=O)c3cc2C...,0.981306
1339,CHEMBL5288032,O=C1Nc2cc3c(-c4ccccc4)c(-c4ccccc4)oc(=O)c3cc2C...,0.884396
1340,CHEMBL5280171,O=C1Oc2cc(OCCN(N3CCOCC3)N3CCOCC3)ccc2C2c3ccccc...,2.175646


In [20]:
dataset_info = dataset_new.copy()

Calculate descriptors using RDkit
a. General molecular descriptors-about 200 molecular descriptors

Calculate descriptors using Mordred 1826 molecular descriptors

In [21]:
def All_Mordred_descriptors(data):
  calc = Calculator(descriptors, ignore_3D=False)
  mols = [Chem.MolFromSmiles(smi) for smi in data]

  #pandas df
  df = calc.pandas(mols)
  return df


In [22]:
mordred_descriptors = All_Mordred_descriptors(dataset_new['SMILES'])

100%|██████████| 1342/1342 [07:04<00:00,  3.16it/s]


In [23]:
mordred_descriptors

,ABC,ABCGG,nAcid,nBase,SpAbs_A,SpMax_A,SpDiam_A,SpAD_A,SpMAD_A,LogEE_A,...,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2
0,21.460493,15.819488,0,2,36.437118,2.30481,4.60962,36.437118,1.301326,4.239949,...,9.934065,62.628628,389.131031,8.279384,2581,38,138.0,155.0,8.277778,6.305556
1,7.140017,6.873356,0,0,12.857279,2.21432,4.428639,12.857279,1.285728,3.188884,...,8.703009,38.792641,137.058912,8.062289,121,11,44.0,48.0,3.722222,2.444444
2,23.581813,16.887365,0,2,40.169989,2.308691,4.617382,40.169989,1.295806,4.336852,...,10.041814,66.126840,433.157246,8.021430,3524,43,152.0,171.0,9.638889,7.055556
3,22.874706,16.590276,0,2,38.895272,2.308217,4.616434,38.895272,1.296509,4.306584,...,10.026811,65.019942,419.141596,8.218463,3180,42,148.0,167.0,9.388889,6.805556
4,22.276989,16.264256,0,2,37.58036,2.307029,4.614058,37.58036,1.295874,4.275237,...,9.998025,63.880417,407.121609,8.662162,2866,40,144.0,162.0,9.138889,6.472222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1337,31.893403,22.117098,0,0,53.313677,2.477573,4.954955,53.313677,1.332842,4.634564,...,10.689692,91.379959,569.119146,9.033637,6263,68,218.0,261.0,11.055556,8.666667
1338,28.248480,20.149874,0,0,46.919272,2.558495,5.116991,46.919272,1.340551,4.514008,...,10.700341,72.129485,478.108420,8.853860,3432,64,196.0,238.0,9.194444,7.500000
1339,28.208040,20.431574,0,0,46.931828,2.560619,5.121238,46.931828,1.340909,4.514124,...,10.716216,72.159409,478.108420,8.853860,3376,65,196.0,239.0,9.194444,7.527778
1340,27.363033,19.173675,0,0,46.405109,2.561591,5.023566,46.405109,1.364856,4.484817,...,10.582003,85.103937,467.205636,7.415962,3741,58,188.0,227.0,7.722222,7.388889


In [24]:
# prompt: concatena mordread_descriptors y dataset_info, pero has que coincidan con los índices

# Make sure the indices of both dataframes are aligned before concatenating
#dataset_info.reset_index(drop=True, inplace=True)
#mordred_descriptors.reset_index(drop=True, inplace=True)

# Concatenate the dataframes
mordred_descriptors = pd.concat([dataset_info, mordred_descriptors], axis=1
                        )

mordred_descriptors

,Cmpd,SMILES,Activity,ABC,ABCGG,nAcid,nBase,SpAbs_A,SpMax_A,SpDiam_A,...,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2
0,CHEMBL377507,C/C(=N/NC(=O)c1ccncc1)c1ccc(N/C(S)=N/c2ccccc2)cc1,5.397940,21.460493,15.819488,0,2,36.437118,2.30481,4.60962,...,9.934065,62.628628,389.131031,8.279384,2581,38,138.0,155.0,8.277778,6.305556
1,CHEMBL64,NNC(=O)c1ccncc1,5.835647,7.140017,6.873356,0,0,12.857279,2.21432,4.428639,...,8.703009,38.792641,137.058912,8.062289,121,11,44.0,48.0,3.722222,2.444444
2,CHEMBL204856,CCOc1ccc(N/C(S)=N\c2ccc(/C(C)=N\NC(=O)c3ccncc3...,6.050610,23.581813,16.887365,0,2,40.169989,2.308691,4.617382,...,10.041814,66.126840,433.157246,8.021430,3524,43,152.0,171.0,9.638889,7.055556
3,CHEMBL204135,COc1ccc(N/C(S)=N\c2ccc(/C(C)=N\NC(=O)c3ccncc3)...,6.327902,22.874706,16.590276,0,2,38.895272,2.308217,4.616434,...,10.026811,65.019942,419.141596,8.218463,3180,42,148.0,167.0,9.388889,6.805556
4,CHEMBL204068,C/C(=N/NC(=O)c1ccncc1)c1ccc(N/C(S)=N/c2ccc(F)c...,6.309804,22.276989,16.264256,0,2,37.58036,2.307029,4.614058,...,9.998025,63.880417,407.121609,8.662162,2866,40,144.0,162.0,9.138889,6.472222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1337,CHEMBL5289448,O=C(NN1C(=O)CSC1c1ccc([N+](=O)[O-])cc1)c1ccc(N...,2.261465,31.893403,22.117098,0,0,53.313677,2.477573,4.954955,...,10.689692,91.379959,569.119146,9.033637,6263,68,218.0,261.0,11.055556,8.666667
1338,CHEMBL5271061,O=C1Nc2cc3c(-c4ccccc4)c(-c4ccccc4)oc(=O)c3cc2C...,0.981306,28.248480,20.149874,0,0,46.919272,2.558495,5.116991,...,10.700341,72.129485,478.108420,8.853860,3432,64,196.0,238.0,9.194444,7.500000
1339,CHEMBL5288032,O=C1Nc2cc3c(-c4ccccc4)c(-c4ccccc4)oc(=O)c3cc2C...,0.884396,28.208040,20.431574,0,0,46.931828,2.560619,5.121238,...,10.716216,72.159409,478.108420,8.853860,3376,65,196.0,239.0,9.194444,7.527778
1340,CHEMBL5280171,O=C1Oc2cc(OCCN(N3CCOCC3)N3CCOCC3)ccc2C2c3ccccc...,2.175646,27.363033,19.173675,0,0,46.405109,2.561591,5.023566,...,10.582003,85.103937,467.205636,7.415962,3741,58,188.0,227.0,7.722222,7.388889


In [25]:
# prompt: descarga mordred_descriptors como un excel

# Assuming mordred_descriptors DataFrame is already created as in your provided code

# Save the Mordred descriptors to an Excel file
#mordred_descriptors.to_excel('mordred_descriptors.xlsx', index=False)
# Save the Mordred descriptors to an Excel file
mordred_descriptors.to_csv('mordred_descriptors_L5.csv', #index=False
                           )